In [ ]:
!python -m pip install keras-tuner==1.4.8

In [3]:
import tensorflow as tf
import numpy as np
import keras_tuner as kt

print("tensorflow:", tf.__version__)
print("numpy:", np.__version__)
print("keras-tuner:", kt.__version__)

tensorflow: 2.18.0
numpy: 1.26.4
keras-tuner: 1.4.8


In [4]:
import sys

# increase recursion limit to prevent potential issues while running complex models
sys.setrecursionlimit(100000)

import libraries

In [ ]:
import keras_tuner as kt
# a linear stack of layers
from tensorflow.keras.models import Sequential 

from tensorflow.keras.layers import Dense, Flatten 
from tensorflow.keras.datasets import mnist
from tensorflow.keras.optimizers import Adam
import os
import warnings

# Suppress all Python warnings
warnings.filterwarnings('ignore')

# Set TensorFlow log level to suppress warnings and info messages
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # 0 = all logs, 1 = filter out INFO, 2 = filter out INFO and WARNING, 3 = ERROR only

load and pre-process MNIST dataset

In [ ]:
(x_train, y_train), (x_val, y_val) = mnist.load_data()
#normalizing pixel values to fall [0,1]
x_train, x_val = x_train / 255.0, x_val / 255.0

print(f'Training data shape: {x_train.shape}')
print(f'Validation data shape: {x_val.shape}')

Training data shape: (60000, 28, 28)
Validation data shape: (10000, 28, 28)


define a function that returns a compiled model that is ready for hyperparameter tuning.

This function builds and compiles a Keras model with hyperparameters:

- hp.Int('units', ...): Defines the number of units in the Dense layer as a hyperparameter.
- hp.Float('learning_rate', ...): Defines the learning rate as a hyperparameter.
- model.compile(): Compiles the model with the Adam optimizer and sparse categorical cross-entropy loss.

In [8]:
# define a model building function

def build_model(hp):
    model = Sequential([
        Flatten(input_shape=(28, 28)),
        Dense(units=hp.Int('units', min_value=32, max_value=512, step=32), activation='relu'),
        Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG')),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

### creating Hyperparameter search tuner

Here we create a RandomSearch tuner and specify the model-building function, optimization objective (validation accuracy), number of trials, and directory for storing results.


- build_model: The model-building function.
- objective='val_accuracy': The metric to optimize (validation accuracy).
- max_trials=10: The maximum number of different hyperparameter configurations to try.
- executions_per_trial=2: The number of times to run each configuration.
- directory='my_dir': Directory to save the results.
- project_name='intro_to_kt': Name of the project for organizing results.
- Epoch = one full pass over the training data.
- Trial = one full training experiment with a specific setting.

In [9]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=2,
    directory='my_dir',
    project_name='intro_to_kt'
)

# Display a summary of the search space 
tuner.search_space_summary()

Search space summary
Default search space size: 2
units (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 32, 'sampling': 'linear'}
learning_rate (Float)
{'default': 0.0001, 'conditions': [], 'min_value': 0.0001, 'max_value': 0.01, 'step': None, 'sampling': 'log'}


### Running the hyperparameter search

Pass in the training data, validation data, and the number of epochs.

In [10]:
# Run the hyperparameter search 
tuner.search(x_train, y_train, epochs=5, validation_data=(x_val, y_val)) 

# Display a summary of the results 
tuner.results_summary() 

Trial 10 Complete [00h 01m 39s]
val_accuracy: 0.9657999873161316

Best val_accuracy So Far: 0.9804499745368958
Total elapsed time: 00h 20m 46s
Results summary
Results in my_dir\intro_to_kt
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 01 summary
Hyperparameters:
units: 512
learning_rate: 0.0015226984419050577
Score: 0.9804499745368958

Trial 08 summary
Hyperparameters:
units: 416
learning_rate: 0.0014679605301420164
Score: 0.9799000024795532

Trial 07 summary
Hyperparameters:
units: 384
learning_rate: 0.0009380326717229366
Score: 0.979200005531311

Trial 00 summary
Hyperparameters:
units: 320
learning_rate: 0.00038233309331929153
Score: 0.9773499965667725

Trial 06 summary
Hyperparameters:
units: 96
learning_rate: 0.0013514233928732255
Score: 0.9764499962329865

Trial 03 summary
Hyperparameters:
units: 160
learning_rate: 0.0004584268052227718
Score: 0.9742999970912933

Trial 04 summary
Hyperparameters:
units: 224
learning_rate: 0.00023205078892812612
Sco

using it

In [11]:
# Retrieve the best hyperparameters 

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0] 
print(f"""The optimal number of units in the first dense layer is {best_hps.get('units')}. The optimal learning rate for the optimizer is {best_hps.get('learning_rate')}.""") 

#  Build and Train the Model with Best Hyperparameters 
model = tuner.hypermodel.build(best_hps) 
model.fit(x_train, y_train, epochs=10, validation_split=0.2) 

# Evaluate the model on the test set 
test_loss, test_acc = model.evaluate(x_val, y_val) 
print(f'Test accuracy: {test_acc}') 

The optimal number of units in the first dense layer is 512. The optimal learning rate for the optimizer is 0.0015226984419050577.
Epoch 1/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - accuracy: 0.9383 - loss: 0.2057 - val_accuracy: 0.9650 - val_loss: 0.1144
Epoch 2/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - accuracy: 0.9746 - loss: 0.0829 - val_accuracy: 0.9734 - val_loss: 0.0844
Epoch 3/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - accuracy: 0.9830 - loss: 0.0547 - val_accuracy: 0.9719 - val_loss: 0.0993
Epoch 4/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - accuracy: 0.9874 - loss: 0.0409 - val_accuracy: 0.9762 - val_loss: 0.0799
Epoch 5/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - accuracy: 0.9895 - loss: 0.0321 - val_accuracy: 0.9747 - val_loss: 0.0859
Epoch 6/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - accuracy: 0.9920 - loss: 0.0257 - val_accuracy: 0.9732 - val_loss: 0.1090
Epoch 7/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.9919 - los